# Benchmark - optimized vs legacy Warp kernels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Forgotten/vlasov-poisson-warp/blob/main/notebooks/03_Benchmark_Optimized_vs_Legacy.ipynb)

Compares the two code paths exposed by `WarpVlasovPoissonSolver`:

* `optimized=False` - 1:1 port of the JAX semi-Lagrangian scheme.
* `optimized=True`  - fused `E + H` v-step kernel and on-device
  cotangent pipeline for the adjoint, with an optional
  `record_history=False` mode that skips per-step `(nx, nv)`
  device-to-host copies.

We measure forward-only and forward+gradient (`jax.grad`) wall
time at several grid sizes.  The two paths are bit-for-bit
equivalent on every output (verified in `tests/test_optimized.py`),
so this notebook is purely about throughput.

**Expected behavior**: the gap is modest on CPU - most of what
the optimized path eliminates is per-step kernel-launch and
host/device transfer overhead, which is cheap on a single CPU
process.  The same code on a CUDA build of Warp should show a
much larger speedup, particularly on the gradient path (which
removes ~4 redundant `(nx, nv)` round-trips per time step) and
in `record_history=False` mode (which removes T full-array
device-to-host copies).


## Setup


In [ ]:
# Colab setup - skip if running locally with the package already installed.
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg,
        ])

for pkg, pip_name in [
    ('warp', 'warp-lang'),
    ('jax', 'jax'),
    ('optax', 'optax'),
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
]:
    _ensure(pkg, pip_name)

# Install the warp_vp_solver package itself.
try:
    import warp_vp_solver  # noqa: F401
except ImportError:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/Forgotten/vlasov-poisson-warp.git@main',
    ])

import warp as wp
wp.init()
print(f'Warp device: {wp.get_preferred_device()}')


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from warp_vp_solver import make_mesh, WarpVlasovPoissonSolver

jax.config.update('jax_enable_x64', True)


## Two-Stream problem at a chosen grid size


In [ ]:
def make_two_stream(nx, nv, *, length_x=10*np.pi, length_v=6.0):
    mesh = make_mesh(length_x, length_v, nx, nv)
    mu = 2.4
    f_eq = (
        np.exp(-0.5*(mesh.V - mu)**2)
        + np.exp(-0.5*(mesh.V + mu)**2)
    ) / (2*np.sqrt(2*np.pi))
    f_iv = (1 + 1e-3*np.cos(0.2*mesh.X)) * f_eq
    return mesh, f_eq, f_iv


## Correctness sanity check


Before timing anything, confirm the two paths really do agree on
the grid we are about to benchmark.  The full test suite checks
this much more thoroughly; this is a one-off guard.


In [ ]:
mesh, f_eq, f_iv = make_two_stream(nx=64, nv=64)
dt = 0.05; t_final = 10*dt
H = np.zeros(mesh.nx)

s_legacy = WarpVlasovPoissonSolver(
    mesh=mesh, dt=dt, f_eq=f_eq, optimized=False,
)
s_fast = WarpVlasovPoissonSolver(
    mesh=mesh, dt=dt, f_eq=f_eq, optimized=True,
)

out_l = s_legacy.run_forward(f_iv, H, t_final)
out_f = s_fast.run_forward(f_iv, H, t_final)
for name, a, b in zip(
    ('f_final', 'f_hist', 'E_hist', 'ee_hist'), out_l, out_f,
):
    np.testing.assert_array_equal(a, b)
print('legacy and optimized paths produce identical outputs.')


## Benchmark helpers


In [ ]:
def bench(fn, warmups=2, repeats=5):
    for _ in range(warmups):
        fn()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times), float(np.mean(times))

def make_runners(nx, nv, *, dt=0.05, t_steps=20):
    mesh, f_eq, f_iv = make_two_stream(nx, nv)
    t_final = t_steps * dt
    H = np.zeros(mesh.nx)
    f_iv_j = jnp.asarray(f_iv); H_j = jnp.asarray(H)
    s_legacy = WarpVlasovPoissonSolver(
        mesh=mesh, dt=dt, f_eq=f_eq, optimized=False,
    )
    s_fast = WarpVlasovPoissonSolver(
        mesh=mesh, dt=dt, f_eq=f_eq, optimized=True,
    )

    def run_legacy_full():
        s_legacy.run_forward(f_iv, H, t_final)
    def run_fast_full():
        s_fast.run_forward(f_iv, H, t_final, record_history=True)
    def run_fast_lite():
        s_fast.run_forward(f_iv, H, t_final, record_history=False)

    def grad_cost_factory(solver):
        def cost(H):
            f_final, _, _, ee = solver.run_forward_jax(
                f_iv_j, H, t_final,
            )
            return jnp.sum(f_final**2) + jnp.sum(ee)
        g = jax.grad(cost)
        def run():
            out = g(H_j)
            jax.block_until_ready(out)
        return run
    return {
        'forward legacy':            run_legacy_full,
        'forward optimized':         run_fast_full,
        'forward optimized (no fhist)': run_fast_lite,
        'grad legacy':               grad_cost_factory(s_legacy),
        'grad optimized':            grad_cost_factory(s_fast),
    }


## Run the benchmark across several grid sizes


In [ ]:
grid_sizes = [(32, 32), (64, 64), (128, 128), (256, 256)]
labels = [
    'forward legacy',
    'forward optimized',
    'forward optimized (no fhist)',
    'grad legacy',
    'grad optimized',
]
results = {label: [] for label in labels}

for nx, nv in grid_sizes:
    print(f'=== grid {nx} x {nv} ===')
    runners = make_runners(nx, nv)
    for label in labels:
        mn, mean = bench(runners[label])
        results[label].append(mn * 1e3)
        print(f'  {label:34s}  min={mn*1e3:7.2f} ms   mean={mean*1e3:7.2f} ms')


## Plot - forward-only timing


In [ ]:
x = np.arange(len(grid_sizes))
labels_x = [f'{nx}x{nv}' for nx, nv in grid_sizes]
fwd_labels = [
    'forward legacy', 'forward optimized', 'forward optimized (no fhist)',
]
width = 0.27
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, label in enumerate(fwd_labels):
    ax.bar(x + (i - 1) * width, results[label], width, label=label)
ax.set_xticks(x); ax.set_xticklabels(labels_x)
ax.set_ylabel('Wall time (ms, min of 5)')
ax.set_title('Forward time loop')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Plot - gradient (forward + adjoint) timing


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
grad_labels = ['grad legacy', 'grad optimized']
width = 0.4
for i, label in enumerate(grad_labels):
    ax.bar(x + (i - 0.5) * width, results[label], width, label=label)
ax.set_xticks(x); ax.set_xticklabels(labels_x)
ax.set_ylabel('Wall time (ms, min of 5)')
ax.set_title('jax.grad through run_forward_jax')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Speedups


In [ ]:
print(f'{"grid":>10}   {"fwd speedup":>14}   {"fwd lite vs legacy":>20}   {"grad speedup":>14}')
for k, (nx, nv) in enumerate(grid_sizes):
    fwd_full = results['forward legacy'][k] / results['forward optimized'][k]
    fwd_lite = results['forward legacy'][k] / results['forward optimized (no fhist)'][k]
    g       = results['grad legacy'][k] / results['grad optimized'][k]
    print(f'{nx}x{nv:<5}   {fwd_full:14.2f}x   {fwd_lite:20.2f}x   {g:14.2f}x')


## Notes

* On CPU (no CUDA) the optimized path mostly removes Python-side
  bookkeeping and small launch overheads.  Speedups of
  ~5-15% are typical here.
* On a CUDA Warp build the per-step transfer / launch elimination
  is much more visible, especially on the gradient path.
* `record_history=False` is the cheapest mode whenever you only
  need `f_final`, `E_hist`, or `ee_hist` (all `make_cost_function_*`
  helpers fall in that category).  It can be wired into the
  optimized JAX wrapper as a future optimization once the adjoint
  is taught to recompute `f_hist` lazily.
